<a href="https://colab.research.google.com/github/aakriti1318/mellea-workshop/blob/main/mellea_rag_context_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Naive RAG vs. Mellea RAG: Validated, Multi-Turn, and Composable

**What you'll build:**

- A small knowledge base indexed with a real semantic retriever: `sentence-transformers` + FAISS
- A **naive RAG** function: stuffed context, one raw LiteLLM completion, no verification
- A typed `AnswerWithCitations` schema that forces every claim to carry a doc id and an exact quote
- A **Mellea RAG** function using `grounding_context`, `format=AnswerWithCitations`, and two custom
  `Requirement`s that check citations in plain Python
- A side-by-side comparison on a query where naive RAG confidently gets it wrong


**Prerequisites:**

- Python 3.11 or 3.12
- [Ollama](https://ollama.com/) installed locally with `ibm/granite4:micro` pulled
- Basic familiarity with Pydantic

Pull the model before running:

```bash
ollama pull ibm/granite4:micro
```


## Step 1. Set up a backend
#### Local Ollama (no credentials required)

Run this cell to install Ollama inside Colab and pull the IBM Granite model.

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Start the Ollama daemon in the background
import subprocess
import time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# Pull the IBM Granite model
! ollama pull ibm/granite4:micro


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.
>>> Cleaning up old version at /usr/local/lib/

## Step 2. Install relevant libraries

Each library here plays a distinct role in the recipe:

- **`mellea[litellm]`**: the Mellea library, with the LiteLLM backend extra installed. Provides `start_session`,
  `instruct()`, `Requirement`, and `RejectionSamplingStrategy` for Part 1, and `ChatContext`, `@generative`, and
  `CBlock` for Part 2.
- **`litellm`**: called directly (bypassing Mellea) for the naive RAG path, so the raw completion and the
  Mellea call go through the same underlying client library.
- **`pydantic`**: defines the `Citation` and `AnswerWithCitations` schema.
- **`faiss-cpu`**: `IndexFlatIP` does the actual nearest-neighbor search over document embeddings.
- **`sentence-transformers`**: generates the embeddings that FAISS indexes and searches over.

Everything in Part 2 (context, `@generative`, `CBlock`) ships in core `mellea` — no extra installs needed.

This is the same retrieval stack used in
[Mellea's own RAG how-to guide](https://github.com/generative-computing/mellea/blob/main/docs/docs/how-to/build-a-rag-pipeline.md).


In [ ]:
%pip install -q "git+https://github.com/ibm-granite-community/utils" "mellea[litellm]" litellm pydantic faiss-cpu sentence-transformers

In [ ]:
%pip install --upgrade Pillow torchvision transformers

## Step 3. Import libraries and set constants

One model constant for generation, one for embeddings, so every path below — naive, Mellea single-shot, and
the Part 2 conversation — always uses the same backend.


In [ ]:
from typing import List

from faiss import IndexFlatIP  # nearest-neighbor search over document embeddings
from litellm import completion  # raw completion call for the naive RAG path
from mellea import start_session, MelleaSession  # Mellea session and its type
from mellea.backends.model_options import ModelOption  # backend-agnostic inference options (seed, temperature)
from mellea.core.requirement import Requirement, ValidationResult  # building a custom, plain-Python requirement
from mellea.stdlib.sampling import RejectionSamplingStrategy  # retry loop when a requirement fails
from pydantic import BaseModel, Field  # typed schema for AnswerWithCitations
from sentence_transformers import SentenceTransformer  # embeds documents and queries for FAISS to search

OLLAMA_MODEL = "ollama/ibm/granite4:micro"
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

print("Generation model:", OLLAMA_MODEL)
print("Embedding model: ", EMBED_MODEL_NAME)


Generation model: ollama/ibm/granite4:micro
Embedding model:  all-MiniLM-L6-v2


## Step 4. The knowledge base

A small set of documents about a fictional product, **Nimbus Cloud**. One document (the last one) describes an
**Enterprise Legacy plan that was discontinued in 2023** — a realistic RAG bug: old documentation that never
got purged from the index, sitting right next to the current docs.

Two test queries:
- `clean_query`: answered directly by one current-plan document
- `trap_query`: asks about the *current* Enterprise plan's uptime guarantee, which is never specified anywhere,
  except the discontinued Legacy plan has a very specific, very citable uptime figure

We'll reuse this exact knowledge base — no changes — for the multi-turn conversation in Part 2.


In [ ]:
docs = [
    "Nimbus Cloud Starter plan: 100 GB storage, single-user only, email support with a 48-hour response time, priced at $5 per month.",
    "Nimbus Cloud Pro plan: 2 TB storage, up to 5 team members, priority email support with a 12-hour response time, priced at $20 per month.",
    "Nimbus Cloud Business plan: 10 TB storage, up to 25 team members, 24/7 chat and email support, priced at $75 per month.",
    "Nimbus Cloud Enterprise plan: unlimited storage, unlimited team members, a dedicated account manager, priced at custom quote only.",
    "All Nimbus Cloud plans include automatic nightly backups and 256-bit AES encryption at rest.",
    "Nimbus Cloud Pro and Business plans support single sign-on (SSO) integration with Okta and Azure AD; Starter does not.",
    "Files deleted from any Nimbus Cloud plan are held in the trash for 30 days before permanent deletion.",
    "Nimbus Cloud does not currently offer a service-level agreement (SLA) or financial compensation for downtime on any plan.",
    "Nimbus Cloud Enterprise Legacy plan (discontinued in 2023): included a 24-hour dedicated-support SLA and a guaranteed 99.5% uptime commitment.",
]

clean_query = "How much storage does the Nimbus Cloud Pro plan include?"
trap_query = "What is the uptime guarantee for the Nimbus Cloud Enterprise plan?"

print(f"Indexed {len(docs)} documents.")
print("Clean query:", clean_query)
print("Trap query: ", trap_query)


Indexed 9 documents.
Clean query: How much storage does the Nimbus Cloud Pro plan include?
Trap query:  What is the uptime guarantee for the Nimbus Cloud Enterprise plan?


## Step 5. Build a semantic retriever

Both RAG paths below share this exact retriever:

- `sentence-transformers` embeds each document into a vector.
- FAISS's `IndexFlatIP` finds the closest matches by inner product, equivalent to cosine similarity for these
  (L2-normalised) embeddings. `IndexFlatIP` ("Flat" = exact, no approximation) is doing a brute-force scan here
  since there are only 9 documents, but the same code scales unchanged to a real, much larger index.
- `search()`'s `k` parameter controls how many documents come back (default `k=3`): too small risks missing the
  one relevant document, too large hands the model more (and more distracting) context to sift through.

Notice the discontinued Legacy plan gets retrieved right alongside the real Enterprise plan document; semantic
similarity search has no notion of "stale."


In [ ]:
def build_index(document_list: List[str], model: SentenceTransformer) -> IndexFlatIP:
    embeddings = model.encode(document_list)
    index = IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)  # type: ignore
    return index


def search(query: str, document_list: List[str], index: IndexFlatIP, model: SentenceTransformer, k: int = 3) -> List[str]:
    query_vector = model.encode([query])
    _, indices = index.search(query_vector, k)
    return [document_list[i] for i in indices[0]]


embed_model = SentenceTransformer(EMBED_MODEL_NAME)
index = build_index(docs, embed_model)

print("Retrieved for the trap query:")
for doc in search(trap_query, docs, index, embed_model):
    print(" -", doc)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieved for the trap query:
 - Nimbus Cloud Enterprise Legacy plan (discontinued in 2023): included a 24-hour dedicated-support SLA and a guaranteed 99.5% uptime commitment.
 - Nimbus Cloud Enterprise plan: unlimited storage, unlimited team members, a dedicated account manager, priced at custom quote only.
 - Nimbus Cloud Starter plan: 100 GB storage, single-user only, email support with a 48-hour response time, priced at $5 per month.


## Step 6. Naive RAG: stuff and hope

The most common real-world RAG pattern: concatenate the retrieved documents into the prompt and ask the model
to answer. There is no check on which document actually supports the answer, and no check for whether the
model even used the right one.

In the `completion()` call:

- `temperature=0` makes the model pick its highest-probability token every time, instead of sampling more
  randomly.
- `seed=42` fixes the random seed.
- Together, they make the completion deterministic: re-running this cell reproduces the same output, and it
  puts this path on equal footing with the Mellea path in Step 11, which uses the same settings.


In [ ]:
def naive_rag(query: str) -> str:
    context_docs = search(query, docs, index, embed_model)
    docs_block = "\n".join(context_docs)
    prompt = f"""Here is some reference material:

{docs_block}

Question: {query}
Answer the question as helpfully as possible."""

    response = completion(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        seed=42,
    )
    return response.choices[0].message.content


print("naive_rag() defined.")
print(f"  model={OLLAMA_MODEL}, temperature=0, seed=42 (deterministic)")
print("  retrieval: search(query, docs, index, embed_model) with default k=3")


naive_rag() defined.
  model=ollama/ibm/granite4:micro, temperature=0, seed=42 (deterministic)
  retrieval: search(query, docs, index, embed_model) with default k=3


## Step 7. Run naive RAG on both queries

Watch the trap query closely: the answer states a specific number with total confidence. Nothing in the output
tells you it actually came from a discontinued plan's spec sheet.


In [ ]:
print("=== CLEAN QUERY ===")
print(naive_rag(clean_query))

print("\n=== TRAP QUERY ===")
print(naive_rag(trap_query))


=== CLEAN QUERY ===
The Nimbus Cloud Pro plan includes 2 TB of storage.

=== TRAP QUERY ===
The uptime guarantee for the Nimbus Cloud Enterprise plan is 99.5%.


The clean query is answered correctly. The trap query is answered *confidently*, and wrong: that uptime
figure belongs to the discontinued Legacy plan, not the current Enterprise plan. Nothing in a plain string
return value tells you that, or lets your code check it.


## Step 8. A typed, cited answer schema

Instead of a free-text string, define exactly what a supported answer looks like:

- `answer`: the answer text itself.
- `citations`: a list of `Citation` objects, each naming the source `doc_id` and quoting the exact `quote` that
  backs the claim; empty if nothing supports the answer.
- `Field(description=...)` on each field isn't just documentation for humans: Mellea sends this schema,
  descriptions included, to the model as part of the instructions for `format=AnswerWithCitations`, so a
  clearer description directly shapes what the model produces.


In [ ]:
class Citation(BaseModel):
    doc_id: str = Field(description="Identifier of the source document, e.g. 'doc0'")
    quote: str = Field(description="Exact text copied verbatim from the document that supports the claim")


class AnswerWithCitations(BaseModel):
    answer: str = Field(description="The answer to the question")
    citations: List[Citation] = Field(
        description="Supporting citations. Empty if the documents do not contain the answer."
    )


print(AnswerWithCitations.model_json_schema())


{'$defs': {'Citation': {'properties': {'doc_id': {'description': "Identifier of the source document, e.g. 'doc0'", 'title': 'Doc Id', 'type': 'string'}, 'quote': {'description': 'Exact text copied verbatim from the document that supports the claim', 'title': 'Quote', 'type': 'string'}}, 'required': ['doc_id', 'quote'], 'title': 'Citation', 'type': 'object'}}, 'properties': {'answer': {'description': 'The answer to the question', 'title': 'Answer', 'type': 'string'}, 'citations': {'description': 'Supporting citations. Empty if the documents do not contain the answer.', 'items': {'$ref': '#/$defs/Citation'}, 'title': 'Citations', 'type': 'array'}}, 'required': ['answer', 'citations'], 'title': 'AnswerWithCitations', 'type': 'object'}


## Step 9. A citation-support requirement

A `Requirement` with a plain Python `validation_fn` runs instead of LLM-as-a-judge: faster, cheaper, and
deterministic. What it checks: every citation's `quote` actually appears, verbatim, in the document it claims
to cite.

- `validation_fn` receives the call's `context`.
- `context.last_output()` returns the most recent model output (a `ModelOutputThunk`).
- `.value` is its raw JSON string, parsed back into `AnswerWithCitations` below.
- `check_only=True` on the `Requirement` means it is validated but never injected into the prompt.


In [ ]:
def make_verbatim_requirement(grounding: dict) -> Requirement:
    def validate(context) -> ValidationResult:
        output = context.last_output()
        try:
            result = AnswerWithCitations.model_validate_json(output.value)
        except Exception as error:
            return ValidationResult(False, reason=f"Output did not match the schema: {error}")

        for citation in result.citations:
            doc_text = grounding.get(citation.doc_id, "")
            if citation.quote.strip().lower() not in doc_text.lower():
                return ValidationResult(
                    False,
                    reason=f"Citation quote {citation.quote!r} does not appear verbatim in {citation.doc_id}.",
                )
        return ValidationResult(True, reason="Every citation is a verbatim quote from its cited document.")

    return Requirement(
        description="Every citation must quote text that actually appears in the referenced document.",
        validation_fn=validate,
        check_only=True,
    )


print("Verbatim citation requirement defined.")
print("Limitation: a quote can be 100% verbatim and still support the wrong claim.")
print("e.g. a real number from a *discontinued* plan, misapplied to the current one.")


Verbatim citation requirement defined.
Limitation: a quote can be 100% verbatim and still support the wrong claim.
e.g. a real number from a *discontinued* plan, misapplied to the current one.


## Step 10. A source-freshness requirement

The trap query exposes exactly that limitation: `"99.5%"` really does appear verbatim in the Legacy plan
document, so verbatim-matching alone would let it through. Add a second, equally simple check: reject
citations whose source document describes a discontinued or legacy offering.


In [ ]:
def make_freshness_requirement(grounding: dict) -> Requirement:
    stale_markers = ("discontinued", "legacy")

    def validate(context) -> ValidationResult:
        output = context.last_output()
        try:
            result = AnswerWithCitations.model_validate_json(output.value)
        except Exception as error:
            return ValidationResult(False, reason=f"Output did not match the schema: {error}")

        for citation in result.citations:
            doc_text = grounding.get(citation.doc_id, "").lower()
            if any(marker in doc_text for marker in stale_markers):
                return ValidationResult(
                    False,
                    reason=(
                        f"Citation {citation.doc_id} is verbatim but describes a discontinued/legacy "
                        "plan, which cannot support a claim about the current plan."
                    ),
                )
        return ValidationResult(True, reason="No citations rely on discontinued or legacy plan documents.")

    return Requirement(
        description="Citations must come from documents describing currently available plans.",
        validation_fn=validate,
        check_only=True,
    )


print("Freshness requirement defined: rejects support drawn from discontinued/legacy plan docs.")


Freshness requirement defined: rejects support drawn from discontinued/legacy plan docs.


## Step 11. Mellea RAG

Start a session, then build the Mellea equivalent of `naive_rag()`. We don't pass a `ctx=` here, so
`start_session()` gives us its default: **`SimpleContext`**. Every `instruct()` call on this session is
independent — no memory of earlier calls. That's exactly right for a one-shot, validation-heavy task like this
one: nothing carries over between the clean and trap queries below, which keeps each call's behavior isolated
and easy to reason about. (Part 2 revisits this choice once we actually want memory.)

In `m.instruct()`:

- `{{question}}` is a Jinja-style placeholder in the instruction string; `user_variables` supplies the value
  that fills it in.
- `grounding_context` carries the retrieved documents, tracked separately from the instruction so Mellea can
  trace exactly what each answer drew on.
- `format=AnswerWithCitations` constrains the output to the typed schema from Step 8.
- `requirements` lists both checks from Steps 9 and 10; they run after every generation.
- `strategy=RejectionSamplingStrategy(loop_budget=3)` retries with a fresh sample, up to 3 times, whenever a
  requirement fails.
- `model_options={ModelOption.SEED: 42, ModelOption.TEMPERATURE: 0}` mirrors the `temperature=0, seed=42` used
  in `naive_rag()` (Step 6), so both paths are equally deterministic and the comparison stays fair.


In [ ]:
# Default: Ollama + Granite locally. No ctx= given -> SimpleContext (independent calls).
m: MelleaSession = start_session(
    backend_name="litellm",
    model_id=OLLAMA_MODEL,
)

# Uncomment for watsonx.ai:
# m = start_session(
#     backend_name="litellm",
#     model_id="watsonx/ibm/granite-4-h-small",
# )

print("Session ready:", m)


=== 09:43:19-INFO ======
Starting Mellea session: backend=litellm, model=ollama/ibm/granite4:micro, context=SimpleContext


INFO:mellea:Starting Mellea session: backend=litellm, model=ollama/ibm/granite4:micro, context=SimpleContext


Session ready: <mellea.stdlib.session.MelleaSession object at 0x7f82e48196a0>


In [ ]:
def mellea_rag(query: str) -> AnswerWithCitations:
    context_docs = search(query, docs, index, embed_model)
    grounding = {f"doc{i}": doc for i, doc in enumerate(context_docs)}

    result = m.instruct(
        "Using the provided documents, answer this question and cite the exact "
        "supporting quote for every factual claim: {{question}}. If the documents do "
        "not contain the answer, say so explicitly and return an empty citations list.",
        user_variables={"question": query},
        grounding_context=grounding,
        format=AnswerWithCitations,
        requirements=[
            make_verbatim_requirement(grounding),
            make_freshness_requirement(grounding),
        ],
        strategy=RejectionSamplingStrategy(loop_budget=3),
        model_options={ModelOption.SEED: 42, ModelOption.TEMPERATURE: 0},
    )
    return AnswerWithCitations.model_validate_json(result.value)


print("mellea_rag() defined.")
print(f"  model={OLLAMA_MODEL}, temperature=0, seed=42 (deterministic, matches naive_rag)")
print("  format=AnswerWithCitations")
print("  requirements=[verbatim citation check, source-freshness check]")
print("  strategy=RejectionSamplingStrategy(loop_budget=3)")


mellea_rag() defined.
  model=ollama/ibm/granite4:micro, temperature=0, seed=42 (deterministic, matches naive_rag)
  format=AnswerWithCitations
  requirements=[verbatim citation check, source-freshness check]
  strategy=RejectionSamplingStrategy(loop_budget=3)


## Step 12. Run Mellea RAG on both queries

Same two queries, same retriever, same underlying model. The only difference is the typed schema and the two
requirements.


In [ ]:
clean_result = mellea_rag(clean_query)
print("=== CLEAN QUERY ===")
print(clean_result.model_dump_json(indent=2))

trap_result = mellea_rag(trap_query)
print("\n=== TRAP QUERY ===")
print(trap_result.model_dump_json(indent=2))


RejectionSamplingStrategy:  33%|███▎      | 1/3 [00:39<01:18, 39.13s/it]

=== 09:43:58-INFO ======
Sampling was successful.



INFO:mellea:Sampling was successful.


=== CLEAN QUERY ===
{
  "answer": "The Nimbus Cloud Pro plan includes 2 TB storage.",
  "citations": [
    {
      "doc_id": "doc0",
      "quote": "2 TB storage"
    }
  ]
}


RejectionSamplingStrategy:   0%|          | 0/3 [00:00<?, ?it/s]

...

=== 09:45:32-INFO ======
FAILED. Valid: 1/2. Failed: 
	 - Citations must come from documents describing currently available plans.


INFO:mellea:FAILED. Valid: 1/2. Failed: 
	 - Citations must come from documents describing currently available plans.
RejectionSamplingStrategy: 100%|██████████| 3/3 [01:34<00:00, 31.46s/it]

=== 09:45:32-INFO ======
Invoking select_from_failure after 3 failed attempts.



INFO:mellea:Invoking select_from_failure after 3 failed attempts.



=== TRAP QUERY ===
{
  "answer": "The uptime guarantee for the Nimbus Cloud Enterprise plan is 99.5%.",
  "citations": [
    {
      "doc_id": "doc0",
      "quote": "included a 24-hour dedicated-support SLA and a guaranteed 99.5% uptime commitment."
    }
  ]
}


Watch the log output above the trap query's result:

- The freshness requirement fails on every attempt (`Citations must come from documents describing currently
  available plans`) because `ibm/granite4:micro` keeps citing the same Legacy document each time.
- `RejectionSamplingStrategy` resamples blindly, with no memory of *why* the previous attempt failed, so a
  small model can keep making the same mistake until the budget runs out.
- Mellea does not silently pretend that didn't happen: it exhausts `loop_budget` and returns the best available
  attempt, still wrong, but now completely legible.
- `trap_result.citations[0].doc_id` and `.quote` tell you exactly what happened, so your code (or a human
  reviewer) can catch it immediately. A bare string can never give you that.

We'll pick this exact failure back up in Part 2's scripted conversation.


### Side-by-side: the trap query

| | Naive RAG | Mellea RAG |
|---|---|---|
| **Return type** | `str` | `AnswerWithCitations` (Pydantic model) |
| **Source visibility** | None: the answer reads as a plain fact | Every claim traces to a `doc_id` + verbatim `quote` |
| **Verbatim fabrication** | Undetectable | Caught by the verbatim-citation requirement |
| **Stale-source misattribution** | Undetectable: the naive answer states the Legacy plan's 99.5% figure as current fact | Flagged on every attempt (visible in the sampling log) |
| **Failure mode** | Silent: reads as confident and correct | Explicit: logged failure reason, and a typed, inspectable result even when retries are exhausted |
| **Downstream code** | Must re-parse and re-verify a string | Can check `answer.citations[i].doc_id` itself, even after a failed repair loop |


## Key Takeaways

**Grounding and citation validation:**

- Naive RAG only reduces hallucination risk; it does not remove it. A stale or mismatched document in the
  retrieved context still produces a confident, wrong answer.
- A raw string return value carries no information about *what* supported the answer, so your code has no way
  to check it.
- `format=AnswerWithCitations` forces every claim to carry a traceable `doc_id` and quote.
- Plain-Python `Requirement`s (`check_only=True`) validate citations deterministically, without an extra LLM
  call — first for verbatim grounding, then for source freshness.
- Verbatim quote-matching alone is not enough: a citation can be perfectly accurate and still support the wrong
  claim. Composing multiple simple requirements catches more than any single one.
- `RejectionSamplingStrategy` retries blindly, with no memory of why the last attempt failed, so a small model
  can fail the same requirement every time and exhaust the budget. Mellea still returns a typed, inspectable
  result instead of masking the failure.

## Next Steps

- **Streaming and Async tutorial**: turn `chat_turn()`'s blocking `m.instruct()`/`m.chat()` calls into
  `ainstruct()`/`achat()`, stream tokens to a UI with `ModelOption.STREAM` + `astream()`, and use
  `wait_for_all_mots` to process a backlog of independent support tickets concurrently — safe because those
  tickets use `SimpleContext`, not the `ChatContext` this lab's live conversation needs.
- [Instruct-Validate-Repair](https://github.com/ibm-granite-community/mellea-cookbook/blob/main/recipes/InstructValidateRepair/InstructValidateRepair.ipynb):
  a deeper look at the requirements-and-retry loop, including `RepairTemplateStrategy`, which feeds the failure
  reason back to the model on retry instead of resampling blindly — worth trying on the trap query above.
- [Quick Start](https://github.com/ibm-granite-community/mellea-cookbook/blob/main/recipes/QuickStart/QuickStart.ipynb):
  Mellea session and `instruct()` basics.
